# 📓 Katalog & Inspeksi Interaktif Asset Google Earth Engine (GEE)
## Riset Tugas Akhir: Deteksi Pola Fenologi Padi Rojolele Srinuk Menggunakan Satelit Multisensor
**Akun Pemilik Asset**: `users/mhakimgf`  
**Wilayah Kajian**: Delanggu & Tulung (Kab. Klaten) serta Tingkir (Kota Salatiga)

Notebook ini berfungsi sebagai buku catatan interaktif yang mendokumentasikan dan memverifikasi seluruh **26 Asset GEE** resmi yang digunakan dalam pipeline pemrosesan citra satelit dan ekstraksi time-series fenologi padi.

In [ ]:
import ee
import geemap
import pandas as pd

# 1. Inisialisasi Google Earth Engine dengan Cloud Project ID
PROJECT_ID = 'ardent-particle-480118-k7'
try:
    ee.Initialize(project=PROJECT_ID)
    print(f"✅ Berhasil terhubung ke GEE dengan Project: {PROJECT_ID}")
except Exception as e:
    print(f"⚠️ Otentikasi GEE diperlukan: {e}")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)

## 📋 Daftar Lengkap 26 Asset GEE `users/mhakimgf`

In [ ]:
assets_catalog = [
    # A. Batas Administrasi & Acuan Lahan Baku
    {"kategori": "Batas/Lahan", "id": "users/mhakimgf/gadm41_IDN", "tipe": "FeatureCollection", "deskripsi": "Batas Administrasi GADM 4.1 Seluruh Indonesia"},
    {"kategori": "Batas/Lahan", "id": "users/mhakimgf/polygon_sawah", "tipe": "FeatureCollection", "deskripsi": "Poligon Acuan Baku Sawah"},
    {"kategori": "Batas/Lahan", "id": "users/mhakimgf/non_sawah", "tipe": "FeatureCollection", "deskripsi": "Poligon Tutupan Non-Sawah (Pemukiman, Industri, Jalan) Delanggu"},
    
    # B. Dataset Final Terlabeli Gabungan
    {"kategori": "Ground Truth Final", "id": "users/mhakimgf/ground_truth_klaten_final", "tipe": "FeatureCollection", "deskripsi": "Ground Truth Klaten Gabungan"},
    {"kategori": "Ground Truth Final", "id": "users/mhakimgf/ground_truth_klaten_final_labeled", "tipe": "FeatureCollection", "deskripsi": "Ground Truth Klaten Terlabeli Lengkap"},
    {"kategori": "Ground Truth Final", "id": "users/mhakimgf/Salatiga/ground_truth_salatiga_final_labeled", "tipe": "FeatureCollection", "deskripsi": "Ground Truth Salatiga Terlabeli Lengkap"},
    
    # C. Rojolele Srinuk (Delanggu, Klaten)
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_1_minggu_200326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 1 Minggu (Genangan / Awal Tanam)"},
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_1_bulan_2_minggu_220326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 1 Bulan 2 Minggu (Vegetatif Aktif)"},
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_1_5_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 1.5 Bulan (Anakan Maksimum)"},
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_2_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 2 Bulan (Inisiasi Malai)"},
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_2_5_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 2.5 Bulan (Heading / Bunting)"},
    {"kategori": "Rojolele Srinuk", "id": "users/mhakimgf/rojolele_srinuk_delanggu_3_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Srinuk Usia 3 Bulan (Pengisian Butir / Pematangan)"},
    
    # D. Varietas Pembanding di Klaten (Delanggu & Tulung)
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/inpari32_delanggu_3_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Delanggu Usia 3 Bulan"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/inpari32_delanggu_3_5_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Delanggu Usia 3.5 Bulan (Siap Panen)"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/inpari32_tulung_1_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tulung Usia 1 Bulan"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/inpari32_tulung_3_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tulung Usia 3 Bulan"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/inpari33_tulung_1_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Inpari 33 Tulung Usia 1 Bulan"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/mapan_tulung_3_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Mapan Tulung Usia 3 Bulan"},
    {"kategori": "Pembanding Klaten", "id": "users/mhakimgf/membramo_delanggu_3_bulan_200326", "tipe": "FeatureCollection", "deskripsi": "Membramo Delanggu Usia 3 Bulan"},
    
    # E. Inpari 32 di Tingkir, Kota Salatiga
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_2_bulan_260326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 2 Bulan (Petak A)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_2_bulan_260326_2", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 2 Bulan (Petak B)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_2_bulan_260326_3", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 2 Bulan (Petak C)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_3_bulan_260326", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 3 Bulan (Petak 1)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_3_bulan_260326_2", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 3 Bulan (Petak 2)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_3_bulan_260326_3", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 3 Bulan (Petak 3)"},
    {"kategori": "Inpari Salatiga", "id": "users/mhakimgf/Salatiga/inpari32_tingkir_3_bulan_260326_4", "tipe": "FeatureCollection", "deskripsi": "Inpari 32 Tingkir Usia 3 Bulan (Petak 4)"}
]

df_assets = pd.DataFrame(assets_catalog)
print(f"Total Asset Terdaftar: {len(df_assets)}")
df_assets

## 🗺️ Visualisasi Peta Interaktif (Leaflet / Geemap)

In [ ]:
# Inisialisasi Peta Geemap terpusat di Delanggu, Klaten
Map = geemap.Map(center=[-7.625, 110.705], zoom=14)
Map.add_basemap("SATELLITE")

try:
    # Memuat Poligon Ground Truth Terlabeli Klaten
    gt_klaten = ee.FeatureCollection("users/mhakimgf/ground_truth_klaten_final_labeled")
    srinuk = gt_klaten.filter(ee.Filter.eq("varietas", "Rojolele Srinuk"))
    non_srinuk = gt_klaten.filter(ee.Filter.neq("varietas", "Rojolele Srinuk"))
    non_sawah = ee.FeatureCollection("users/mhakimgf/non_sawah")
    
    # Menambahkan layer ke peta
    Map.addLayer(srinuk.style(color="00FF00", fillColor="00FF0044", width=2), {}, "Rojolele Srinuk (Hijau)")
    Map.addLayer(non_srinuk.style(color="FFA500", fillColor="FFA50044", width=2), {}, "Padi Non-Srinuk (Oranye)")
    Map.addLayer(non_sawah.style(color="FF0000", fillColor="FF000044", width=1), {}, "Non-Sawah (Merah)")
    print("✅ Layer berhasil dimuat ke peta interaktif.")
except Exception as e:
    print(f"Info: {e}")

Map